# Multimodal Food Concept Embedding Space

This notebook inspects the canonical-food multimodal embedding space. The current default embedding is an offline structured vector built from nutrients, OpenFoodFacts product/processing fields, FoodAtlas/FooDB chemistry, HMDB metabolomics, disease/pathway graph summaries, and stable hashed text-evidence features.

It also includes an optional section for LLM/API sentence embeddings from the generated food concept sentences.

In [ ]:
from pathlib import Path
import json
import os

import pandas as pd
import plotly.express as px

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
OUT = ROOT / 'outputs'
EMBEDDINGS = OUT / 'embeddings'
EMBEDDINGS.mkdir(parents=True, exist_ok=True)

import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

## Build Or Refresh The Offline Structured Space

This regenerates the offline structured embedding, 2D projection, food concept sentences, cluster summaries, and compact-hover HTML visualization.

In [ ]:
from diet_data_enhancement.multimodal import build_all

summary = build_all()
summary

## Load Outputs

In [ ]:
vectors = pd.read_parquet(EMBEDDINGS / 'canonical_food_multimodal_vectors.parquet')
projection = pd.read_csv(EMBEDDINGS / 'embedding_space_projection.csv')
sentences = pd.read_csv(EMBEDDINGS / 'canonical_food_concept_sentences.csv')
dimensions = pd.read_csv(EMBEDDINGS / 'canonical_food_multimodal_vector_dimensions.csv')
patterns = pd.read_csv(EMBEDDINGS / 'food_disease_pattern_discovery.csv')

vectors.shape, projection.shape, sentences.shape, dimensions.shape, patterns.shape

## Where Do The Categories Come From?

`canonical_category` is not inferred by the embedding model. It is inherited from the HPP food table category during canonicalization. Canonical IDs are created from normalized food name plus normalized category, so the category is part of the canonical food identity. The cluster ID is different: it is computed from the multimodal vector.

In [ ]:
canonical = pd.read_csv(OUT / 'canonical' / 'canonical_foods.csv')
category_counts = (
    canonical.groupby('canonical_category', dropna=False)
    .agg(canonical_foods=('canonical_food_id', 'nunique'), hpp_food_ids=('hpp_food_count', 'sum'))
    .sort_values('canonical_foods', ascending=False)
    .reset_index()
)
category_counts.head(20)

## Compact-Hover Visualization

Hover text is intentionally short: canonical food name and assigned category only.

In [ ]:
from diet_data_enhancement.multimodal import _categorical_embedding_figure

fig = _categorical_embedding_figure(
    projection,
    'Multimodal Canonical Food Concept Space',
    'PC1 of structured multimodal vector',
    'PC2 of structured multimodal vector',
)
fig.write_html(EMBEDDINGS / 'embedding_space_interactive.html', include_plotlyjs='cdn', full_html=True)
fig

## Inspect One Food Concept

In [ ]:
def inspect_food(search='coffee'):
    hits = sentences[sentences['canonical_name'].str.contains(search, case=False, na=False)].copy()
    hits = hits.merge(projection, on=['canonical_food_id', 'canonical_name', 'canonical_category'], how='left')
    cols = [
        'canonical_food_id', 'canonical_name', 'canonical_category', 'cluster_id',
        'foodb_compound_count', 'foodatlas_compound_count', 'hmdb_metabolite_count',
        'hmdb_disease_count', 'hmdb_pathway_count', 'concept_sentence'
    ]
    return hits[cols]

inspect_food('coffee')

## First-Pass Food-Disease And Pathway Patterns

These are descriptive cluster summaries, not causal disease claims and not formal enrichment tests yet.

In [ ]:
patterns.head(30)

## Optional LLM/API Sentence Embeddings

The default structured embedding above is offline and reproducible. To try LLM/API embeddings, set `OPENAI_API_KEY` in the terminal or paste the key when prompted. The key is not saved by the project.

Command-line equivalent:

```bash
conda run --no-capture-output -n ds python -m diet_data_enhancement.pipeline build-llm-sentence-embeddings
```

Optional model override:

```bash
OPENAI_FOOD_CONCEPT_EMBEDDING_MODEL=text-embedding-3-small conda run --no-capture-output -n ds python -m diet_data_enhancement.pipeline build-llm-sentence-embeddings
```

In [ ]:
if os.getenv('OPENAI_API_KEY'):
    from diet_data_enhancement.multimodal import build_llm_sentence_embeddings
    llm_summary = build_llm_sentence_embeddings()
    display(llm_summary)
else:
    print('OPENAI_API_KEY is not set. Set it in the terminal or run the command above and paste the key when prompted.')

## View LLM Sentence Embedding Space If Available

In [ ]:
llm_projection_path = EMBEDDINGS / 'llm_embedding_space_projection.csv'
if llm_projection_path.exists():
    llm_projection = pd.read_csv(llm_projection_path)
    from diet_data_enhancement.multimodal import _categorical_embedding_figure
    fig_llm = _categorical_embedding_figure(
        llm_projection,
        'LLM Sentence Embedding Food Concept Space',
        'PC1 of sentence embedding',
        'PC2 of sentence embedding',
    )
    fig_llm.show()
else:
    print('No LLM sentence embedding projection found yet.')